# SQL Analysis: Credit Risk Portfolio

This notebook answers the project's business questions **using SQL only**. 
It uses **SQLite** (built into Python, no server to install). The same queries run on PostgreSQL / MySQL / SQL Server with one small change: replace `MIN(a, b)` with `LEAST(a, b)`.

### How it is organised
1. **Setup**: load the CSV into a database table, then create two *views*:
   - `credit_clean`: applies the same cleaning rules as the Python notebook
   - `credit_banded`: adds readable bands (age band, utilization band, …)
2. **Analysis**: one section per business question. Every query is also saved as a stand-alone script in the `sql/` folder.

> **SQL concepts used** (with explanations next to each query): `GROUP BY`, `CASE WHEN`, `HAVING`, CTEs (`WITH`), window functions (`NTILE`, `RANK`, `SUM() OVER`), `COALESCE`.

## 1. Setup: load the data into a SQL database

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200

def find_data_dir():
    """Locate cs-training.csv wherever the notebook is run from.
    Looks in the project layout first (../data/raw), then the current folder and its neighbours.
    If only GiveMeSomeCredit.zip is found, it is unzipped automatically."""
    here = Path.cwd()
    folders = [here / "../data/raw", here / "data/raw", here, here / "..", here / "data", here / "../data"]
    for folder in folders:
        if (folder / "cs-training.csv").exists():
            return folder.resolve()
    for folder in folders:                                    # fall back to the original Kaggle zip
        zip_file = folder / "GiveMeSomeCredit.zip"
        if zip_file.exists():
            import zipfile
            target = folder.resolve() / "GiveMeSomeCredit_unzipped"
            zipfile.ZipFile(zip_file).extractall(target)
            print("Unzipped", zip_file.name, "->", target)
            return target
    raise FileNotFoundError(
        "Could not find cs-training.csv. Put cs-training.csv (and cs-test.csv) in a 'data/raw' folder "
        "next to this notebook's folder, or in the same folder as the notebook. Current folder: " + str(here))

DATA_DIR = find_data_dir()
CSV_PATH = DATA_DIR / "cs-training.csv"
print("Using data file:", CSV_PATH)

# Save the database in the project layout if it exists, otherwise next to the notebook
DB_FOLDER = Path("../data/processed") if (Path.cwd() / "../data/raw").exists() else Path("outputs")
DB_FOLDER.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_FOLDER / "credit_risk.db"

# Shorter column names (same mapping as the Python notebook)
COLUMN_NAMES = {
    "SeriousDlqin2yrs": "defaulted", "RevolvingUtilizationOfUnsecuredLines": "utilization", "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "late_30_59", "DebtRatio": "debt_ratio", "MonthlyIncome": "income",
    "NumberOfOpenCreditLinesAndLoans": "open_lines", "NumberOfTimes90DaysLate": "late_90",
    "NumberRealEstateLoansOrLines": "real_estate", "NumberOfTime60-89DaysPastDueNotWorse": "late_60_89",
    "NumberOfDependents": "dependents",
}

connection = sqlite3.connect(DB_PATH)

raw = pd.read_csv(CSV_PATH, index_col=0).rename(columns=COLUMN_NAMES)
raw.index.name = "borrower_id"
raw.to_sql("credit_raw", connection, if_exists="replace")          # creates the table credit_raw

def run_query(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, connection)

run_query("SELECT COUNT(*) AS rows_loaded FROM credit_raw")

Using data file: /home/claude/proj/data/raw/cs-training.csv


,rows_loaded
0,150000


### Create the cleaning and banding views
A **view** is a saved query that behaves like a table, so we write the cleaning logic once and every later query reuses it.

In [2]:
setup_sql = """-- Raw table is loaded from cs-training.csv by the notebook (table name: credit_raw)
--   borrower_id, defaulted, utilization, age, late_30_59, debt_ratio, income,
--   open_lines, late_90, real_estate, late_60_89, dependents

DROP VIEW IF EXISTS credit_clean;
CREATE VIEW credit_clean AS
SELECT
    borrower_id,
    defaulted,
    age,
    -- 96 / 98 are placeholder codes, not real counts -> flag them and reset the count to 0
    CASE WHEN late_30_59 >= 96 OR late_60_89 >= 96 OR late_90 >= 96 THEN 1 ELSE 0 END AS delinq_code_flag,
    CASE WHEN late_30_59 >= 96 THEN 0 ELSE late_30_59 END AS late_30_59,
    CASE WHEN late_60_89 >= 96 THEN 0 ELSE late_60_89 END AS late_60_89,
    CASE WHEN late_90    >= 96 THEN 0 ELSE late_90    END AS late_90,
    MIN(utilization, 2.0)                                AS utilization,   -- cap at 200%
    CASE WHEN income IS NULL THEN NULL ELSE MIN(income, 50000) END AS income,  -- cap outliers
    -- debt ratio is only credible when income is known and the value is plausible
    CASE WHEN income IS NULL OR debt_ratio > 5 THEN NULL ELSE debt_ratio END AS debt_ratio,
    open_lines,
    real_estate,
    dependents
FROM credit_raw
WHERE age >= 18;                                   -- removes the impossible age-0 record

-- NOTE: MIN(a, b) with two arguments is SQLite's "smaller of". Use LEAST(a, b) in PostgreSQL / MySQL.

DROP VIEW IF EXISTS credit_banded;
CREATE VIEW credit_banded AS
SELECT
    c.*,
    CASE WHEN age <= 25 THEN '1: 18-25'  WHEN age <= 35 THEN '2: 26-35'  WHEN age <= 45 THEN '3: 36-45'
         WHEN age <= 55 THEN '4: 46-55'  WHEN age <= 65 THEN '5: 56-65'  ELSE '6: 65+' END       AS age_band,
    CASE WHEN income IS NULL  THEN '7: Unknown'  WHEN income <= 2500 THEN '1: <2.5k'
         WHEN income <= 4000  THEN '2: 2.5-4k'   WHEN income <= 5500 THEN '3: 4-5.5k'
         WHEN income <= 7500  THEN '4: 5.5-7.5k' WHEN income <= 10000 THEN '5: 7.5-10k'
         ELSE '6: 10k+' END                                                                       AS income_band,
    CASE WHEN utilization <= 0.10 THEN '1: 0-10%'   WHEN utilization <= 0.30 THEN '2: 10-30%'
         WHEN utilization <= 0.50 THEN '3: 30-50%'  WHEN utilization <= 0.75 THEN '4: 50-75%'
         WHEN utilization <= 1.00 THEN '5: 75-100%' ELSE '6: >100%' END                           AS utilization_band,
    CASE WHEN debt_ratio IS NULL THEN '6: Unknown'  WHEN debt_ratio <= 0.2 THEN '1: 0-0.2'
         WHEN debt_ratio <= 0.4  THEN '2: 0.2-0.4'  WHEN debt_ratio <= 0.6 THEN '3: 0.4-0.6'
         WHEN debt_ratio <= 1.0  THEN '4: 0.6-1.0'  ELSE '5: 1.0-5.0' END                         AS debt_ratio_band,
    CASE WHEN open_lines <= 2 THEN '1: 0-2'  WHEN open_lines <= 5  THEN '2: 3-5'
         WHEN open_lines <= 8 THEN '3: 6-8'  WHEN open_lines <= 12 THEN '4: 9-12' ELSE '5: 13+' END AS open_lines_band,
    CASE WHEN late_90 > 0 OR late_60_89 > 0 THEN '3: 60+ days late'
         WHEN late_30_59 > 0                THEN '2: 30-59 days late only'
         ELSE '1: Never late' END                                                                 AS past_due_history
FROM credit_clean AS c;"""
connection.executescript(setup_sql)
run_query("SELECT * FROM credit_banded LIMIT 5")

,borrower_id,defaulted,age,delinq_code_flag,late_30_59,late_60_89,late_90,utilization,income,debt_ratio,open_lines,real_estate,dependents,age_band,income_band,utilization_band,debt_ratio_band,open_lines_band,past_due_history
0,1,1,45,0,2,0,0,0.77,"9,120.00",0.80,13,6,2.00,3: 36-45,5: 7.5-10k,5: 75-100%,4: 0.6-1.0,5: 13+,2: 30-59 days late only
1,2,0,40,0,0,0,0,0.96,"2,600.00",0.12,4,0,1.00,3: 36-45,2: 2.5-4k,5: 75-100%,1: 0-0.2,2: 3-5,1: Never late
2,3,0,38,0,1,0,1,0.66,"3,042.00",0.09,2,0,0.00,3: 36-45,2: 2.5-4k,4: 50-75%,1: 0-0.2,1: 0-2,3: 60+ days late
3,4,0,30,0,0,0,0,0.23,"3,300.00",0.04,5,0,0.00,2: 26-35,2: 2.5-4k,2: 10-30%,1: 0-0.2,2: 3-5,1: Never late
4,5,0,49,0,1,0,0,0.91,"50,000.00",0.02,7,1,0.00,4: 46-55,6: 10k+,5: 75-100%,1: 0-0.2,3: 6-8,2: 30-59 days late only


## 2. Data-quality checks

Before analysing anything, count the problems in the *raw* table. These checks justify the cleaning rules in `credit_clean`.

*Script: `sql/01_data_quality_checks.sql`*

**Query 1: Row count and missing values**

`COUNT(col)` ignores NULLs while `COUNT(*)` counts every row, so the difference is the number of missing values.

In [3]:
run_query("""
SELECT COUNT(*)                          AS total_rows,
       COUNT(*) - COUNT(income)          AS missing_income,
       COUNT(*) - COUNT(dependents)      AS missing_dependents,
       ROUND(100.0 * (COUNT(*) - COUNT(income)) / COUNT(*), 1) AS missing_income_pct
FROM credit_raw;
""")

,total_rows,missing_income,missing_dependents,missing_income_pct
0,150000,29731,3924,19.80


**Query 2: Impossible or suspicious values**

Each `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` is a conditional count: how many rows break a rule.

In [4]:
run_query("""
SELECT SUM(CASE WHEN age < 18 THEN 1 ELSE 0 END)                                         AS age_under_18,
       SUM(CASE WHEN late_30_59 >= 96 OR late_60_89 >= 96 OR late_90 >= 96 THEN 1 ELSE 0 END)   AS placeholder_codes_96_98,
       SUM(CASE WHEN utilization > 2 THEN 1 ELSE 0 END)                                         AS utilization_over_200pct,
       SUM(CASE WHEN income IS NULL OR debt_ratio > 5 THEN 1 ELSE 0 END)                        AS unreliable_debt_ratio
FROM credit_raw;
""")

,age_under_18,placeholder_codes_96_98,utilization_over_200pct,unreliable_debt_ratio
0,1,269,371,32083


**Query 3: Are the 96/98 'placeholder' rows special?**

Their default rate is about 8x the rest, so we keep them (flagged) rather than delete them.

In [5]:
run_query("""
SELECT CASE WHEN late_30_59 >= 96 OR late_60_89 >= 96 OR late_90 >= 96
            THEN 'Placeholder code (96/98)' ELSE 'Normal rows' END AS row_type,
       COUNT(*)                                  AS borrowers,
       ROUND(100.0 * AVG(defaulted), 1)          AS default_rate_pct
FROM credit_raw
GROUP BY row_type;
""")

,row_type,borrowers,default_rate_pct
0,Normal rows,149731,6.60
1,Placeholder code (96/98),269,54.60


## 3. Q1 - What percentage of borrowers default?

The headline numbers for the whole portfolio. `AVG(defaulted)` works because `defaulted` is 0/1, so its average *is* the default rate.

*Script: `sql/02_portfolio_overview.sql`*

**Query 1: Overall default rate**

In [6]:
run_query("""
SELECT COUNT(*)                          AS borrowers,
       SUM(defaulted)                    AS defaults,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_clean;
""")

,borrowers,defaults,default_rate_pct
0,149999,10026,6.68


**Query 2: Portfolio profile: defaulters vs non-defaulters**

Side-by-side averages show how the two groups differ at a glance.

In [7]:
run_query("""
SELECT CASE defaulted WHEN 1 THEN 'Defaulted' ELSE 'Did not default' END AS outcome,
       COUNT(*)                          AS borrowers,
       ROUND(AVG(age), 1)                AS avg_age,
       ROUND(AVG(income), 0)             AS avg_income,
       ROUND(AVG(utilization), 2)        AS avg_utilization,
       ROUND(AVG(late_30_59 + late_60_89 + late_90), 2) AS avg_late_events
FROM credit_clean
GROUP BY defaulted;
""")

,outcome,borrowers,avg_age,avg_income,avg_utilization,avg_late_events
0,Did not default,139973,52.80,"6,542.00",0.30,0.29
1,Defaulted,10026,45.90,"5,551.00",0.71,2.00


## 4. Q2 - Which customer characteristics are associated with default?

Same pattern each time: `GROUP BY` a band, count borrowers, and compute the default rate. `credit_banded` already contains the bands.

*Script: `sql/03_default_by_characteristics.sql`*

**Query 1: Default rate by age band**

In [8]:
run_query("""
SELECT age_band,
       COUNT(*)                          AS borrowers,
       SUM(defaulted)                    AS defaults,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_banded
GROUP BY age_band
ORDER BY age_band;
""")

,age_band,borrowers,defaults,default_rate_pct
0,1: 18-25,3027,338,11.17
1,2: 26-35,18458,2053,11.12
2,3: 36-45,29819,2628,8.81
3,4: 46-55,36690,2786,7.59
4,5: 56-65,33406,1531,4.58
5,6: 65+,28599,690,2.41


**Query 2: Default rate by payment history**

In [9]:
run_query("""
SELECT past_due_history,
       COUNT(*)                          AS borrowers,
       SUM(defaulted)                    AS defaults,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_banded
GROUP BY past_due_history
ORDER BY past_due_history;
""")

,past_due_history,borrowers,defaults,default_rate_pct
0,1: Never late,119906,3411,2.84
1,2: 30-59 days late only,17213,2063,11.99
2,3: 60+ days late,12880,4552,35.34


**Query 3: Default rate by number of 90+ day lates**

`MIN(late_90, 4)` lumps '4 or more' together.

In [10]:
run_query("""
SELECT MIN(late_90, 4)                  AS times_90_days_late,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_clean
GROUP BY MIN(late_90, 4)
ORDER BY times_90_days_late;
""")

,times_90_days_late,borrowers,default_rate_pct
0,0,141930,4.72
1,1,5243,33.66
2,2,1555,49.90
3,3,667,57.72
4,4,604,66.06


**Query 4: Default rate by number of dependents**

In [11]:
run_query("""
SELECT COALESCE(CAST(CAST(MIN(dependents, 4) AS INTEGER) AS TEXT), 'Unknown') AS dependents_4_means_4plus,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_clean
GROUP BY dependents_4_means_4plus
ORDER BY dependents_4_means_4plus;
""")

,dependents_4_means_4plus,borrowers,default_rate_pct
0,0,86902,5.86
1,1,26316,7.35
2,2,19521,8.11
3,3,9483,8.83
4,4,3853,10.28
5,Unknown,3924,4.56


## 5. Q3 - Does loan amount (credit exposure) affect default?

The dataset has **no loan-amount column**, so we use the closest measures of how much credit someone carries: utilization, debt ratio and number of open lines.

*Script: `sql/04_credit_exposure.sql`*

**Query 1: Default rate by credit utilization**

In [12]:
run_query("""
SELECT utilization_band,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_banded
GROUP BY utilization_band
ORDER BY utilization_band;
""")

,utilization_band,borrowers,default_rate_pct
0,1: 0-10%,64404,1.81
1,2: 10-30%,28478,3.14
2,3: 30-50%,15830,5.85
3,4: 50-75%,13764,10.13
4,5: 75-100%,24202,18.21
5,6: >100%,3321,37.25


**Query 2: Default rate by debt ratio**

Debt ratio = monthly debt payments / monthly income. 'Unknown' = no income reported.

In [13]:
run_query("""
SELECT debt_ratio_band,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_banded
GROUP BY debt_ratio_band
ORDER BY debt_ratio_band;
""")

,debt_ratio_band,borrowers,default_rate_pct
0,1: 0-0.2,40667,6.09
1,2: 0.2-0.4,38284,5.63
2,3: 0.4-0.6,21521,7.69
3,4: 0.6-1.0,12563,10.63
4,5: 1.0-5.0,4881,12.99
5,6: Unknown,32083,5.51


**Query 3: Default rate by number of open credit lines**

In [14]:
run_query("""
SELECT open_lines_band,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_banded
GROUP BY open_lines_band
ORDER BY open_lines_band;
""")

,open_lines_band,borrowers,default_rate_pct
0,1: 0-2,12992,12.92
1,2: 3-5,33598,6.68
2,3: 6-8,39420,5.35
3,4: 9-12,36305,5.92
4,5: 13+,27684,6.67


## 6. Q4 - How does income relate to repayment?

Includes a **window function** (`NTILE`) to cut borrowers into five equal-sized income groups.

*Script: `sql/05_income_analysis.sql`*

**Query 1: Default rate by income band**

In [15]:
run_query("""
SELECT income_band,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_banded
GROUP BY income_band
ORDER BY income_band;
""")

,income_band,borrowers,default_rate_pct
0,1: <2.5k,18111,8.68
1,2: 2.5-4k,22287,9.22
2,3: 4-5.5k,21871,7.63
3,4: 5.5-7.5k,22246,6.20
4,5: 7.5-10k,17434,5.10
5,6: 10k+,18319,4.32
6,7: Unknown,29731,5.61


**Query 2: Default rate by income quintile (NTILE)**

`NTILE(5) OVER (ORDER BY income)` ranks borrowers by income and splits them into 5 groups of equal size. The inner query (a *CTE*, the `WITH` block) does the ranking; the outer query summarises it.

In [16]:
run_query("""
WITH ranked AS (
    SELECT defaulted, income,
           NTILE(5) OVER (ORDER BY income) AS income_quintile      -- 1 = lowest income
    FROM credit_clean
    WHERE income IS NOT NULL
)
SELECT income_quintile,
       COUNT(*)                          AS borrowers,
       ROUND(MIN(income), 0)             AS min_income,
       ROUND(MAX(income), 0)             AS max_income,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM ranked
GROUP BY income_quintile
ORDER BY income_quintile;
""")

,income_quintile,borrowers,min_income,max_income,default_rate_pct
0,1,24054,0.00,"3,000.00",9.05
1,2,24054,"3,000.00","4,544.00",8.65
2,3,24054,"4,545.00","6,300.00",7.01
3,4,24053,"6,300.00","9,083.00",5.53
4,5,24053,"9,083.00","50,000.00",4.50


**Query 3: Is a missing income a warning sign?**

In [17]:
run_query("""
SELECT CASE WHEN income IS NULL THEN 'Income not reported' ELSE 'Income reported' END AS income_status,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM credit_clean
GROUP BY income_status;
""")

,income_status,borrowers,default_rate_pct
0,Income not reported,29731,5.61
1,Income reported,120268,6.95


## 7. Q5 - Which customer segments have the highest risk?

A segment = payment history x utilization group x age group. We then rank segments and measure how much of the total default volume each one holds.

*Script: `sql/06_risk_segmentation.sql`*

**Query 1: Segment risk league table**

Step by step:
1. `segmented` assigns each borrower to a segment.
2. `segment_stats` counts borrowers and defaults per segment.
3. The final `SELECT` adds **window functions**: `SUM(...) OVER ()` gives portfolio totals, so we can compute each segment's share and 'lift' (risk vs average); `RANK() OVER (ORDER BY ...)` ranks the segments.
4. `HAVING` removes tiny segments (under 300 borrowers) whose rates are unreliable.

In [18]:
run_query("""
WITH segmented AS (
    SELECT defaulted,
           past_due_history,
           CASE WHEN utilization <= 0.30 THEN 'Low util (<30%)'
                WHEN utilization <= 0.75 THEN 'Medium util (30-75%)'
                ELSE 'High util (>75%)' END AS utilization_group,
           CASE WHEN age < 35 THEN 'Under 35' WHEN age < 55 THEN '35-54' ELSE '55+' END AS age_group
    FROM credit_banded
),
segment_stats AS (
    SELECT past_due_history, utilization_group, age_group,
           COUNT(*)       AS borrowers,
           SUM(defaulted) AS defaults
    FROM segmented
    GROUP BY past_due_history, utilization_group, age_group
    HAVING COUNT(*) >= 300
)
SELECT RANK() OVER (ORDER BY 1.0 * defaults / borrowers DESC)                    AS risk_rank,
       past_due_history, utilization_group, age_group,
       borrowers,
       ROUND(100.0 * defaults / borrowers, 1)                                    AS default_rate_pct,
       ROUND(100.0 * defaults / SUM(defaults) OVER (), 1)                        AS share_of_all_defaults_pct,
       ROUND((1.0 * defaults / borrowers) / (1.0 * SUM(defaults) OVER () / SUM(borrowers) OVER ()), 2) AS lift_vs_average
FROM segment_stats
ORDER BY risk_rank
LIMIT 12;
""")

,risk_rank,past_due_history,utilization_group,age_group,borrowers,default_rate_pct,share_of_all_defaults_pct,lift_vs_average
0,1,3: 60+ days late,High util (>75%),35-54,3818,46.10,17.60,6.90
1,2,3: 60+ days late,High util (>75%),Under 35,1725,44.60,7.70,6.68
2,3,3: 60+ days late,High util (>75%),55+,1507,40.90,6.20,6.13
3,4,3: 60+ days late,Medium util (30-75%),Under 35,437,32.70,1.40,4.90
4,5,3: 60+ days late,Medium util (30-75%),35-54,1440,30.80,4.40,4.60
5,6,3: 60+ days late,Medium util (30-75%),55+,848,25.80,2.20,3.86
6,7,2: 30-59 days late only,High util (>75%),Under 35,942,24.50,2.30,3.67
7,8,3: 60+ days late,Low util (<30%),Under 35,408,24.00,1.00,3.59
8,9,2: 30-59 days late only,High util (>75%),35-54,2592,23.00,6.00,3.45
9,10,3: 60+ days late,Low util (<30%),35-54,1451,20.00,2.90,2.99


**Query 2: Heat-map data: payment history x age band**

Ready to paste into a heat-map: one row per combination.

In [19]:
run_query("""
SELECT past_due_history, age_band,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * AVG(defaulted), 1)  AS default_rate_pct
FROM credit_banded
GROUP BY past_due_history, age_band
ORDER BY past_due_history, age_band;
""")

,past_due_history,age_band,borrowers,default_rate_pct
0,1: Never late,1: 18-25,2468,6.30
1,1: Never late,2: 26-35,13668,5.30
2,1: Never late,3: 36-45,22574,3.80
3,1: Never late,4: 46-55,28232,3.10
4,1: Never late,5: 56-65,27473,1.90
5,1: Never late,6: 65+,25491,1.00
6,2: 30-59 days late only,1: 18-25,233,21.50
7,2: 30-59 days late only,2: 26-35,2269,15.60
8,2: 30-59 days late only,3: 36-45,4050,13.80
9,2: 30-59 days late only,4: 46-55,4931,12.40


## 8. Bonus - A simple rules-based risk tier (no machine learning)

Many lenders start with transparent business rules. Here is one that can be written in SQL, followed by how well it separates good and bad borrowers. It is a useful benchmark for the ML model (see the Python notebook).

*Script: `sql/07_rule_based_risk_tiers.sql`*

**Query 1: Rule-based tiers and their actual default rates**

Rules are checked top to bottom; the first match wins.

In [20]:
run_query("""
WITH tiered AS (
    SELECT defaulted,
           CASE WHEN late_90 > 0 OR late_60_89 > 0 OR utilization > 1.0 THEN 'Tier 4 - High risk'
                WHEN late_30_59 > 0 OR utilization > 0.75             THEN 'Tier 3 - Elevated'
                WHEN utilization > 0.30 OR age < 35                   THEN 'Tier 2 - Moderate'
                ELSE 'Tier 1 - Low risk' END AS risk_tier
    FROM credit_clean
)
SELECT risk_tier,
       COUNT(*)                          AS borrowers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio,
       SUM(defaulted)                    AS defaults,
       ROUND(100.0 * AVG(defaulted), 2)  AS default_rate_pct
FROM tiered
GROUP BY risk_tier
ORDER BY risk_tier;
""")

,risk_tier,borrowers,pct_of_portfolio,defaults,default_rate_pct
0,Tier 1 - Low risk,74637,49.80,831,1.11
1,Tier 2 - Moderate,29488,19.70,1127,3.82
2,Tier 3 - Elevated,31101,20.70,3091,9.94
3,Tier 4 - High risk,14773,9.80,4977,33.69


**Query 2: Cumulative defaults captured as we review tiers from riskiest down**

A running total (`SUM() OVER (ORDER BY ...)`) shows how many defaults we find if we only look at the riskiest tiers.

In [21]:
run_query("""
WITH tiered AS (
    SELECT defaulted,
           CASE WHEN late_90 > 0 OR late_60_89 > 0 OR utilization > 1.0 THEN 'Tier 4 - High risk'
                WHEN late_30_59 > 0 OR utilization > 0.75             THEN 'Tier 3 - Elevated'
                WHEN utilization > 0.30 OR age < 35                   THEN 'Tier 2 - Moderate'
                ELSE 'Tier 1 - Low risk' END AS risk_tier
    FROM credit_clean
),
tier_stats AS (
    SELECT risk_tier, COUNT(*) AS borrowers, SUM(defaulted) AS defaults
    FROM tiered GROUP BY risk_tier
)
SELECT risk_tier, borrowers, defaults,
       ROUND(100.0 * SUM(borrowers) OVER (ORDER BY risk_tier DESC) / SUM(borrowers) OVER (), 1) AS cum_pct_of_borrowers,
       ROUND(100.0 * SUM(defaults)  OVER (ORDER BY risk_tier DESC) / SUM(defaults)  OVER (), 1) AS cum_pct_of_defaults
FROM tier_stats
ORDER BY risk_tier DESC;
""")

,risk_tier,borrowers,defaults,cum_pct_of_borrowers,cum_pct_of_defaults
0,Tier 4 - High risk,14773,4977,9.80,49.60
1,Tier 3 - Elevated,31101,3091,30.60,80.50
2,Tier 2 - Moderate,29488,1127,50.20,91.70
3,Tier 1 - Low risk,74637,831,100.00,100.00


## Summary of SQL findings

| Question | Result (from the queries above) |
|---|---|
| Q1 Default rate | **6.68 %** (10,026 of 149,999 borrowers) |
| Q2 Characteristics | Payment history is by far the strongest: 2.8 % (never late) vs 35.3 % (60+ days late). Younger borrowers default more (11 % under 36 vs 2.4 % over 65). |
| Q3 Exposure | Utilization climbs from 1.8 % to 37 % (over limit); debt ratio above 0.6 doubles risk; thin files (0-2 lines) default at ~13 %. |
| Q4 Income | Lowest-income quintile defaults at 9.1 %, highest at 4.5 %. Missing income is *not* riskier (5.6 %). |
| Q5 Segments | *60+ days late + high utilization* segments default at 41-46 %, about 6-7x average. |
| Bonus | A simple four-tier rule set (no ML) puts 9.8 % of borrowers in its top tier, capturing 49.6 % of defaults, and 80.5 % of defaults within the top 30.6 %. The ML model does better (top 10 % captures 55 %; see the Python notebook), but the rules are a strong, transparent baseline. |

**Next:** the Python notebook adds machine-learning models, and the Tableau dashboard presents the results.

In [22]:
connection.close()